# MGnipy — Capabilities Demo

This notebook demonstrates everything the library can do **right now**, and explicitly marks what is broken or not yet implemented. It is organized as a live tour before the PyPI draft publish.

**Sections:**
1. Setup
2. `MGnifier` — direct low-level API (works)
3. Output formatters: `to_df`, `to_polars`, `to_json`, `to_list` (works)
4. Query planning: `dry_run`, `preview`, `explain` (works)
5. Immutable filter cloning (works)
6. Pagination: `page(n)`, `get(limit)` (works)
7. `MGnipy` facade + proxy classes (partially works)
8. Async: `aget`, `apage` (works)
9. ❌ What is currently broken
10. 🗺️ What comes next

---
## 1. Setup

In [ ]:
# Verify installation
import mgnipy
print(f"mgnipy version: {mgnipy.__version__}")

In [ ]:
from mgnipy.V2.core import MGnifier
from mgnipy.V2.query_set import QuerySet
from mgnipy import MGnipy
print("All imports OK")

---
## 2. `MGnifier` — direct low-level API

`MGnifier` is the core query object. It wraps `QuerySet` (which does query building) and delegates HTTP calls to `QueryExecutor`. You can use it directly or through the higher-level `MGnipy` facade.

In [ ]:
# Create a query for studies — no API call yet
mg = MGnifier(resource="studies", params={"page_size": 5})
print(mg)

In [ ]:
# Fetch just the first page — one API call
mg.first()
print(f"Pages fetched so far: {list(mg._results.keys())}")

---
## 3. Output formatters

All four formatters work once data is in `_results`.

In [ ]:
# pandas DataFrame
df = mg.to_df()
print(f"to_df() → {type(df).__name__}, shape: {df.shape}")
df.head()

In [ ]:
# Polars DataFrame
pl_df = mg.to_polars()
print(f"to_polars() → {type(pl_df).__name__}, shape: {pl_df.shape}")
pl_df.head()

In [ ]:
# List of dicts
records = mg.to_list()
print(f"to_list() → {type(records).__name__}, length: {len(records)}")
print(f"First record keys: {list(records[0].keys()) if records else 'none'}")

In [ ]:
# JSON string (newline-delimited by default)
json_str = mg.to_json()
print(f"to_json() → {type(json_str).__name__}, {len(json_str)} chars")
print(json_str[:200], "...")

---
## 4. Query planning: `dry_run`, `preview`, `explain`

Before fetching everything you can inspect the plan — how many records, how many pages, which URLs.

In [ ]:
# dry_run: makes one small API call (page_size=1) to learn total count, then prints the plan
planner = MGnifier(resource="analyses", params={"page_size": 10})
planner.dry_run()

In [ ]:
# After dry_run, count and total_pages are populated
print(f"Total records: {planner.count}")
print(f"Total pages (at page_size=10): {planner.total_pages}")

In [ ]:
# explain: print the first N request URLs without making them
planner.explain(head=3)

In [ ]:
# list_urls: returns the full URL list
urls = planner.list_urls()
print(f"{len(urls)} URLs total. First: {urls[0]}")

In [ ]:
# preview: fetches page 1 and returns a DataFrame immediately
preview_df = MGnifier(resource="samples", params={"page_size": 5}).preview()
print(f"preview() → {type(preview_df).__name__}, shape: {preview_df.shape}")
preview_df.head()

---
## 5. Immutable filter cloning

`filter()` returns a **new** `QuerySet` with the updated params — the original is untouched.
This makes it safe to build queries incrementally.

In [ ]:
base = QuerySet(resource="studies")
filtered = base.filter(biome_lineage="root:Environmental:Aquatic", page_size=5)

print(f"base params:     {base.params}")
print(f"filtered params: {filtered.params}")
print(f"Same object?     {base is filtered}")

In [ ]:
# Chain filters — each step is a new clone
qs = (
    QuerySet(resource="studies")
    .filter(biome_lineage="root:Environmental")
    .page_size(3)
)
print(f"Chained params: {qs.params}")
print(f"Request URL: {qs.request_url}")

In [ ]:
# Fetch the filtered results
qs.first()
df = qs.to_df()
print(f"Rows returned: {len(df)}")
df.head()

---
## 6. Pagination: `page(n)` and `get(limit)`

Pages are fetched individually and cached. Already-fetched pages are not re-requested.

In [ ]:
pg = MGnifier(resource="biomes", params={"page_size": 5})
pg.dry_run()
print(f"Total pages: {pg.total_pages}")

In [ ]:
# Fetch page 1
pg.page(1)
print(f"Page 1 in results: {pg._is_in_results(1)}")
print(f"Page 2 in results: {pg._is_in_results(2)}")

In [ ]:
# Fetch page 3 (skipping page 2 — non-contiguous fetch is fine)
pg.page(3)
print(f"Pages fetched: {sorted(pg._results.keys())}")
print(f"DataFrame has {len(pg.to_df())} rows (2 pages × 5 per page)")

In [ ]:
# get(limit=N) fetches however many pages are needed to satisfy limit records
# Requires dry_run() first (or pass safety=False to skip)
limited = MGnifier(resource="samples", params={"page_size": 5})
limited.dry_run()
limited.get(limit=12)  # will fetch 3 pages (5+5+5 = 15, enough for 12)
print(f"Records retrieved: {len(limited.to_df())} (asked for 12, got nearest page boundary)")

---
## 7. `MGnipy` facade + proxy classes

`MGnipy` is the top-level entry point. It uses `__getattr__` to dispatch attribute access to typed proxy classes.

**⚠️ Known limitation:** the config passed to `MGnipy()` is not forwarded to the proxy. Custom base URLs and auth tokens are silently ignored until M2 is fixed.

In [ ]:
client = MGnipy()
print(f"Available resources: {client.list_resources()}")

In [ ]:
# Accessing a resource returns a typed proxy (list-type)
studies_proxy = client.studies
print(type(studies_proxy))
print(studies_proxy)

In [ ]:
# Proxies expose the same query-building API as MGnifier
filtered_proxy = studies_proxy.filter(biome_lineage="root:Environmental", page_size=3)
print(f"Filter returned new object: {filtered_proxy is not studies_proxy}")
print(f"Filtered params: {filtered_proxy.params}")

In [ ]:
# Fetch first page through the proxy
filtered_proxy.first()
df = filtered_proxy.to_df()
print(f"Rows: {len(df)}")
df.head()

In [ ]:
# Biomes has a special tree visualisation (after fetching)
biomes = client.biomes
biomes.first()
print(f"Biome lineages: {biomes.lineages[:5]}")

In [ ]:
# Config bug demonstration — custom base_url is silently ignored right now
from mgnipy._models.config import MgnipyConfig
default_url = str(MgnipyConfig().base_url)

custom_client = MGnipy(base_url="https://custom.example.com")
proxy = custom_client.studies

print(f"Custom URL given to MGnipy: https://custom.example.com")
print(f"URL actually used by proxy: {proxy._base_url}")
print(f"Bug present: {str(proxy._base_url) == default_url}  ← should be False after M2 fix")

---
## 8. Async: `aget`, `apage`, `afirst`

Every sync method has an async counterpart. Use these when you need to concurrently fetch many resources.

In [ ]:
import asyncio

async def demo_async():
    mg = MGnifier(resource="runs", params={"page_size": 5})
    await mg.afirst()
    df = mg.to_df()
    print(f"Async fetch → {len(df)} rows")
    return df

# In Jupyter, use await directly (event loop already running)
df_async = await demo_async()
df_async.head()

In [ ]:
async def demo_concurrent_pages():
    mg = MGnifier(resource="samples", params={"page_size": 10})
    mg.dry_run()
    # aget fetches all pages concurrently (with semaphore to protect the server)
    await mg.aget(limit=30, safety=False)
    return mg.to_df()

df_concurrent = await demo_concurrent_pages()
print(f"Concurrent fetch → {len(df_concurrent)} rows")

---
## 9. ❌ What is currently broken

These cells document bugs and missing features. They are **expected to fail** until the corresponding milestone is fixed.

### M1 — `cli.py` is missing

In [ ]:
# This should succeed after M1 is fixed
try:
    import mgnipy.cli
    print(f"✅ mgnipy.cli imported, main={mgnipy.cli.main}")
except ModuleNotFoundError as e:
    print(f"❌ M1 not fixed: {e}")
    print("   Fix: create mgnipy/cli.py with a main() function")

### M2 — Config not passed to proxies

In [ ]:
# After M2 is fixed, proxy._base_url should match the custom URL
from mgnipy import MGnipy
from mgnipy._models.config import MgnipyConfig

custom = MGnipy(base_url="https://staging.example.com")
proxy = custom.studies

expected = "https://staging.example.com"
actual = str(proxy._base_url)

if actual == expected:
    print("✅ M2 fixed: config flows through")
else:
    print(f"❌ M2 not fixed: proxy uses '{actual}' instead of '{expected}'")
    print("   Fix: mgnipy/mgnipy.py:52 — pass base_url to proxy constructor")

### Not-yet-implemented: `SingleResource` (accession lookup)

In [ ]:
# The plan (Day 2) calls for: mgnipy.studies["MGYS00001422"] → lazy SingleResource
# Currently __getitem__ on the proxy requires data to already be fetched
from mgnipy.V2.proxies import Studies

s = Studies()
try:
    item = s["MGYS00001422"]
    print(f"✅ Accession lookup returned: {type(item).__name__}")
except (AttributeError, KeyError, TypeError) as e:
    print(f"❌ SingleResource not implemented: {type(e).__name__}: {e}")
    print("   Fix: implement SingleResource class and update proxy __getitem__")

### Not-yet-implemented: `.order_by()` and `.exists()`

In [ ]:
from mgnipy.V2.query_set import QuerySet

qs = QuerySet(resource="studies")

for method in ("order_by", "exists"):
    if hasattr(qs, method):
        print(f"✅ {method}() exists")
    else:
        print(f"❌ {method}() not implemented yet")

### Not-yet-implemented: `describe_resources()`

In [ ]:
from mgnipy import MGnipy

client = MGnipy()
result = client.describe_resources()

if result is not None:
    print(f"✅ describe_resources() returned: {result}")
else:
    print("❌ describe_resources() is a stub — returns None")
    print("   Fix: implement in mgnipy/mgnipy.py")

---
## 10. 🗺️ What comes next

### Milestones for the 2-hour PyPI publish session

| # | Fix | File | Time |
|---|-----|------|------|
| M1 | Create `cli.py` with `main()` | `mgnipy/cli.py` (new) | 15 min |
| M2 | Pass config to proxy constructors | `mgnipy/mgnipy.py:52` | 10 min |
| M3 | Narrow `testpaths` in pytest config | `pyproject.toml` | 5 min |
| M4 | Build wheel and test-install | — | 15 min |
| M5 | Rewrite README with accurate examples | `README.md` | 20 min |
| M6 | Add CHANGELOG | `CHANGELOG.md` (new) | 5 min |
| M7 | Tag version and push | git | 10 min |

### To run the milestone tests

```bash
# All milestones (offline, no API calls)
uv run pytest tests/milestones/test_milestones.py -v

# Include live-API regression tests
uv run pytest tests/milestones/test_milestones.py -v -m live_api

# After fixing M1, remove @pytest.mark.xfail from TestM1_CLI.test_after_fix_*
# and re-run — those tests should now be green
```

### Deferred post-publish

- `SingleResource` — lazy accession-keyed objects (`studies["MGYS00001422"]`)
- `.order_by()`, `.exists()` methods
- `describe_resources()` implementation
- >85% test coverage with mocked API calls
- Full docstring pass